In [55]:

import ifcopenshell as ifc
import ifcopenshell.util as ifc_util
import ifcopenshell.util.element as ifc_element
import ifcopenshell.util.placement as ifc_placement
import ifcopenshell.util.geolocation as ifc_geolocator
import glob 
import numpy as np
import pandas as pd
import ifcopenshell.geom
import ifcopenshell.util.shape
import random

from typing import Dict, Any, Optional, Literal
from unittest import case

print("ifcopenshell version: ", ifc.__version__)

ifcopenshell version:  0.9.0.post1


In [56]:
path = glob.glob("BIM/*.ifc")
print(path[-1])
ifc_model = ifc.open(path[-1])
print(ifc_model.schema)
# for i in range(1,10):
#     random_entity = ifc_model.by_id(i)
#     print(random_entity.get_info())
    
# entity = ifc_model.by_type("IfcSIUnit")
# len(entity)
# walls = ifc_model.by_type("IfcWall")
# len(walls)
# print (walls[0].is_a("IfcWallStandardCase"))
# print (walls[0].is_a("IfcWall"))
# print (walls[0].id())
# print (walls[0].is_a("IfcWindow"))
# print (walls[0].is_a("IfcElement"))


BIM/edited.ifcview.ifc
IFC4


In [57]:
measuring_device = ifc_model.by_guid("17ivOSUPDAPhmNRfJI$rhR")
measuring_device.Tag = "Gadget"
measuring_device.get_info()
measuring_device.is_a("IfcElement")
measuring_device.get_info()

{'id': 11286,
 'type': 'IfcBuildingElementProxy',
 'GlobalId': '17ivOSUPDAPhmNRfJI$rhR',
 'OwnerHistory': None,
 'Name': 'X_ACCX_FT13_CEST04_0_Messpunkt',
 'Description': None,
 'ObjectType': None,
 'ObjectPlacement': #11296=IfcLocalPlacement(#91,#11295),
 'Representation': #11307=IfcProductDefinitionShape($,$,(#11306)),
 'Tag': 'Gadget',
 'PredefinedType': None}

In [58]:
# EXTRACT ALL MEASURING DEVICES WHICH STARTS WITH "G_"

gs = []
unique_measures = set()

for e in ifc_model.by_type("IfcElement"):
    try:
        name = e.Name
        if "_Messpunkt" in name:
            if name.startswith("G_"):
               e.Name = name.replace("_Messpunkt", "")
               gs.append(e)       
               unique_measures.add(e.Name.split("_")[1]) 
    except:
        pass

print("Number of measuring devices: ", len(gs))
print(unique_measures)

Number of measuring devices:  74
{'NSSN', 'KKFT', 'NGSX', 'NGSY', 'NGSF', 'NGST', 'TEMP', 'LADS', 'DMSD', 'DMSK', 'KKFF', 'ACCZ', 'IWAR', 'PYRS'}


In [59]:
all_elements = ifc_model.by_type("IfcElement")
len(all_elements)
ifc_element.get_psets(ifc_model.by_id(12860))


{'Sensorinformation': {'01 Produktbezeichnung': 'NA',
  '02 Status': 'eingebaut',
  '03 Kasten': 'Kasten Kistler',
  '04 Zusatz': 'NA',
  '05 Bemerkung': 'NA',
  '06 Laenge': 'NA',
  '07 Ansprechpartner': 'Kistler Instrumente',
  '08 DatumAnfang': '17.09.2025',
  '09 DatumEnde': 'NA',
  'id': 21515}}

In [60]:
df = pd.DataFrame(columns=["Name","QoI", "Units", "IDP", "GlobalId", "Type", "AbsoluteCoordinates [X, Y, Z]", "RelativeCoordinates [X, Y, Z]"])

In [61]:
# UNITS {sensor_type: unit}
units = {
 'DMSD': {"unit": "μm/m", "QoI": "Strain"},
 'PYRS': {"unit": "W/m²", "QoI": "Solar Radiation"},
 'KKFT': {"unit": "°C", "QoI": "Temperature"},
 'KKFF': {"unit": "%", "QoI": "Humidity"},
 'IWAR': {"unit": "mm", "QoI": "Displacement"},
 'ACCZ': {"unit": "m/s²", "QoI": "Acceleration"},
 'LADS': {"unit": "m", "QoI": "Distance"},
 'NGSF': {"unit": "%", "QoI": "Moisture"},
 'NGST': {"unit": "°C", "QoI": "Temperature"},
 'TEMP': {"unit": "°C", "QoI": "Temperature"},
 'DMSK': {"unit": "μm/m", "QoI": "Strain"},
 'NGSX': {"unit": "mm/m", "QoI": "Tilt"},
 'NSSN': {"unit": "unknown", "QoI": "unknown"},
 'NGSY': {"unit": "mm/m", "QoI": "Tilt"},
 'ACCX': {"unit": "m/s²", "QoI": "Acceleration"},
 }

{'DMSD', 'PYRS', 'KKFT', 'KKFF', 'IWAR', 'ACCZ', 'LADS', 'NGSF', 'NGST', 'TEMP', 'DMSK', 'NGSX', 'NSSN', 'NGSY'}

# SMALL TRANSLATION DICTIONARY
# ... the english and german sensor labels differ   -\_**_/-

# ... de - en
# ... TF - PE
# ... BU - CB

small_dicts = {
    1: {"NGSX": "TILX", "NGSY": "TILY"}, 
    2: {"TF": "PE"},
    3: {"BU": "CB"}
}

In [62]:
settings = ifcopenshell.geom.settings()
settings.set("use-world-coords", True)

def get_element_absolute_coordinates(element):
    shape = ifcopenshell.geom.create_shape(settings, element)
    vertices = ifcopenshell.util.shape.get_vertices(shape.geometry)
    if len(vertices) > 0:
        coordinates = (vertices.min(axis=0) + vertices.max(axis=0)) / 2
        return np.round(coordinates, 3)
    else:
        return None

def get_element_relative_coordinates(element, full_coordinates=False):
    if element.ObjectPlacement:
        coordinates = ifc_placement.get_local_placement(element.ObjectPlacement)
        if full_coordinates:
            return np.round(coordinates, 3)
        else:
            return np.round(coordinates[:3,3], 3)
    else:
        return None


def gen_random_idp(existing_idps):
        return str(random.randint(1000, 9999))
        

IDPs = []
def gen_sofistik_idp(global_id):
    idp = str(global_id[:4])
    if idp not in IDPs:
        IDPs.append(idp)
    else:
        # gen random idp
        print(f"IDP {idp} already exists, generating a new one...")
        while True:
            idp = gen_random_idp(IDPs)
            if idp not in IDPs:
                IDPs.append(idp)
                break
    return idp

for e in gs:
    # coordinates = get_element_coordinates(e, True)
    relative_coordinates = get_element_relative_coordinates(e, False)
    absolute_coordinates = get_element_absolute_coordinates(e)
    code = e.Name.split("_")[1]
    idp = gen_sofistik_idp(e.get_info().get("GlobalId", None))

    df = pd.concat([df, pd.DataFrame([[
        e.get_info().get("Name", None),
        units.get(code, {}).get("QoI", None),
        units.get(code, {}).get("unit", None),
        idp,
        e.get_info().get("GlobalId", None), 
        e.is_a(), absolute_coordinates, relative_coordinates
        ]], columns=df.columns)], ignore_index=True)
    
len(df)
df.head(10)
df.to_csv("IFC_Sensor_Coordinates.csv", index=False)

In [ ]:
# compute the coordinates for automatic axis placements of the sensors in the SOFiSTiK model.
reference_point = [0, 0, 0]
...

In [63]:
# Create SOFiSTIK input file for the sensors

with open("SOFiSTIK_Sensor_Input.dat", "w") as f:
    f.write("+prog sofimshc\n")
    f.write("HEAD Sensor Definition\n")    
    f.write("!The IDP of sensors is a 4-digit code derived from the GlobalId of the IFC element (as the first 4 characters - if unique, otherwise generated randomly).\n")
    f.write("!Tilrmeters - the title always corresponds to the gadget label in GERMAN labeling (please... get rid of the redundant DE/EN labels... keep just one :D)\n")
    
    
    def write_sensortypes(df: pd.DataFrame, TYPE: Optional[Literal["S", "J", "A", "E", "H"]] = None, header: str = "", cadinp_lang: str = "DE") -> None:
        # df.sort_values(by="Name", key=lambda x: x.str.split("_")[2], inplace=True)
        df = df.copy()  # Create a copy of the DataFrame to avoid modifying the original
        def key_func(col):
            # Split the string by "_" and extract the third part
            third_part = col.str.split("_").str[2]
            # Extract the last two characters and convert to integer
            last_two_chars = third_part.str[-2:].str[::-1].astype(int)
            return last_two_chars

        df.sort_values(
            by="Name",
            key=key_func,
            inplace=True
        )
        
        if header:
            f.write(f"!{header}\n")
        for index, row in df.iterrows():
            # Get the position from the label
            name = row["Name"]
            span = name.split("_")[2][-2]
            girder = name.split("_")[2][-1]

            # TODO: CHECK WHETHER THIS HOLDS !!!
            reference_point = ["#A10", "#A20", "#A30"][int(span) - 1]  # Assuming span is 1, 2, or 3
            # SENSOR OFFSET FROM THE REFERENCE POINT (e.g., the girder's origin) - this is a placeholder, you need to compute it based on your specific requirements
            offset = float(name.split("_")[3][2:])/100  # Assuming the offset is the fourth part of the name



            position = reference_point + "+" + str(offset)  # commpute from the label

            comment = "! Sensor: " + row['Name'] + " - " + row['QoI'] + " - " + row['Units']
            match cadinp_lang:
                case "DE":
                    tp = "" if TYPE is None else f"TYP {TYPE} "
                    f.write(f"GAXP A{girder} IDS 0 S {position}\t{tp} GRP ... NQR ... SPT ... IDP '{row['IDP']}' BEZ '{row['Name']}' {comment}\n")
                case "EN":
                    tp = "" if TYPE is None else f"TYPE {TYPE} "
                    f.write(f"GAXP A{girder} IDS 0 S {position}\t{tp} GRP ... NCS ... SPT ... IDP '{row['IDP']}' TITL '{row['Name']}' {comment}\n")
                case _:
                    raise ValueError(f"Unsupported cadinp_lang: {cadinp_lang}, supported values are 'DE' and 'EN'. If you want to work with the openLAB model, use 'DE', because SOFiSTiK can not interpret 'EN' keywords when the original is in german.")
    
    write_sensortypes(df[df["QoI"] == "Tilt"][df[df["QoI"] == "Tilt"]["Name"].str.contains("_FT")], header="Tiltmeters")
    write_sensortypes(df[df["QoI"] == "Strain"], header="Strain gauges")
    write_sensortypes(df[df["QoI"] == "Acceleration"], header="Acceleration sensors")

    # for index, row in df[df["QoI"] == "Tilt"].iterrows():
    #     f.write(f"GAXP {}")
    # for axis in ["A1", "A2", "A3"]:

    

    # ...
    f.write("END")
